## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

## TODAY:

- Part A: We will divide our documents into CHUNKS
- Part B: We will encode our CHUNKS into VECTORS and put in Chroma
- Part C: We will visualize our vectors

### PART A: Divide our documents into chunks

In [ ]:
import os
import glob
import tiktoken
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from supabase_db import ingest, retrieve, rag, fetch_vectors
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

In [ ]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4.1-nano"

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
from supabase import create_client

load_dotenv()
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client("https://mgxdfusfutmnsyqzcekn.supabase.co", SUPABASE_KEY)

In [ ]:
try:
    result = supabase.table("documents").select("id").limit(1).execute()
    print("Connected:", result.data)
except Exception as e:
    print("Connection failed:", e)

In [ ]:
# How many characters in all the documents?

knowledge_base_path = "../../knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

In [ ]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

In [ ]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("../../knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

In [ ]:
documents[1]

In [ ]:
# --- Usage ---
ingest(documents)

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter
query = "Who is Avery?"
chunks = retrieve(query)

In [ ]:
chunks[1]

In [ ]:
# --- GENERATE: answer using retrieved context ---
answer = rag(query)
print(answer)

### Part B: Visualize!

In [ ]:
# --- 2. Reduce to 3D with t-SNE ---

def run_tsne(vectors: np.ndarray, perplexity: int = 5) -> np.ndarray:
    perplexity = min(perplexity, len(vectors) - 1)
    tsne = TSNE(n_components=3, perplexity=perplexity, random_state=42, init="pca")
    return tsne.fit_transform(vectors)

In [ ]:
# --- 3. Plot ---

def plot_tsne_3d(contents, sources, coords):
    df = pd.DataFrame({
        "x": coords[:, 0], "y": coords[:, 1], "z": coords[:, 2],
        "label": contents, "category": sources
    })

    categories = df["category"].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(categories)))
    color_map = dict(zip(categories, colors))

    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection="3d")

    for category, group in df.groupby("category"):
        ax.scatter(group["x"], group["y"], group["z"],
                   label=category, color=color_map[category], alpha=0.7, s=80)

    ax.set_title("Document Embeddings (t-SNE 3D)")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.set_zlabel("t-SNE 3")
    ax.legend(title="doc_type", bbox_to_anchor=(1.05, 1))
    plt.tight_layout()
    plt.savefig("tsne_3d.png", dpi=150)
    plt.show()

In [ ]:
contents, sources, vectors = fetch_vectors()
coords = run_tsne(vectors)
plot_tsne_3d(contents, sources, coords)

In [ ]:
import plotly.express as px

df = pd.DataFrame({
    "x": coords[:, 0], "y": coords[:, 1], "z": coords[:, 2],
    "content": contents, "category": sources
})

fig = px.scatter_3d(df, x="x", y="y", z="z",
                    color="category", hover_data=["content"],
                    title="Document Embeddings (t-SNE 3D)")
fig.show()

In [ ]:
query = "What is Avery's salary?"
answer = rag(query)
print(answer)

In [ ]:
query = "And what does Avery do ?"
answer = rag(query)
print(answer)


In [ ]:
query = "Who is Michelle ?"
answer = rag(query)
print(answer)

In [ ]:
query = "What is Michelle's salary?"
answer = rag(query)
print(answer)